# Does learning matter? Trained vs untrained AlexNet
**Reading:** Cichy, R. M., Khosla, A., Pantazis, D., Torralba, A., & Oliva, A. (2016). Comparison of deep neural networks to spatio-temporal cortical dynamics of human visual object recognition reveals hierarchical correspondence. *Scientific Reports, 6*, 27755.

Cichy et al. compared AlexNet-like networks with human brain responses. As a control, they also tested an **untrained** network: the same architecture, but with random weights. Today we do the same: we compare a **trained** and an **untrained** AlexNet.

**What you will do**
1. Load a trained and an untrained AlexNet
2. Classify two photos with both networks
3. Compare their first-layer feature maps
4. Build **similarity matrices**: how similar does each network find photos and toys?

## Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import torch
from torchvision import models, transforms
device = "cpu"

In [ ]:
# Find the course data in Sciebo (works even if the shared folder was renamed)
matches = list(Path.home().glob("sciebo/*/Week4/"))
if not matches:
    raise FileNotFoundError(
        "Course data not found. Did you connect Sciebo when starting the server, "
        "and accept the shared CNU folder in Sciebo?"
    )
DATA = matches[0]
IMAGE_DIR = DATA / "images"                       # photos and toys for today
WEEK3_DIR = DATA.parent / "Week3" / "images"      # we reuse two photos from Week 3
print(f"Using data from {DATA}")

## 1. Load a trained and an untrained AlexNet
Both networks have exactly the same **architecture** (the same layers). The only difference is the **weights**:
- the **trained** network learned its weights from 1.2 million ImageNet photos,
- the **untrained** network has **random** weights, as before any training.

In [ ]:
weights = models.AlexNet_Weights.IMAGENET1K_V1
categories = weights.meta["categories"]

# Trained: the ImageNet weights (as in Week 2 and 3)
trained = models.alexnet(weights=weights).to(device).eval()

# Untrained: same architecture, random weights.
# torch.manual_seed fixes the random numbers, so everyone gets the same "random" network.
torch.manual_seed(0)
untrained = models.alexnet(weights=None).to(device).eval()

networks = {"trained": trained, "untrained": untrained}

## 2. How images are prepared (as in Week 2 and 3)

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
what_network_sees = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224)])

def classify(model, imgs, k=5):
    """Return the k most likely labels and their probabilities for each image."""
    batch = torch.stack([preprocess(img) for img in imgs]).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(batch), dim=1)
    top_probs, top_idx = probs.topk(k, dim=1)
    return [[(categories[i], p.item()) for p, i in zip(p_row, i_row)]
            for p_row, i_row in zip(top_probs, top_idx)]

def show_top5(images, names, results, title=None):
    """One row per image: the image on the left, its top-5 predictions on the right."""
    n = len(images)
    fig, axes = plt.subplots(n, 2, figsize=(9, 2.3 * n), gridspec_kw={"width_ratios": [1, 1.7]})
    axes = np.atleast_2d(axes)
    for row, (img, name, res) in enumerate(zip(images, names, results)):
        ax_img, ax_bar = axes[row]
        ax_img.imshow(what_network_sees(img)); ax_img.axis("off")
        ax_img.set_title(name, fontsize=10)
        labels = [label for label, _ in res][::-1]
        probs  = [p for _, p in res][::-1]
        ax_bar.barh(labels, probs, color="tab:blue")
        ax_bar.set_xlim(0, 1)
        for y, p in enumerate(probs):
            ax_bar.text(p + 0.01, y, f"{p:.2f}", va="center", fontsize=9)
    axes[-1, 1].set_xlabel("probability")
    if title:
        fig.suptitle(title)
    plt.tight_layout(); plt.show()

## 3. Classify two photos with both networks
These are the zebra and plane photos from Week 3. The trained network recognised both with high confidence.

**Before running:** what do you expect from the untrained network?

In [ ]:
photo_names = ["zebra_photo", "plane_photo"]
photos = [Image.open(next(WEEK3_DIR.glob(name + ".*"))).convert("RGB") for name in photo_names]

for net_name, model in networks.items():
    show_top5(photos, photo_names, classify(model, photos), title=f"{net_name} AlexNet")

**Think about it:**
- What does the untrained network answer? How confident is it?
- Both networks have the same layers. So what exactly did the trained network learn?

## 4. First-layer feature maps: trained vs untrained
As in Week 2, we record the output of the first convolutional layer with a **hook** and look at the feature maps.

In [ ]:
def conv1_maps(model, img):
    """Return the 64 feature maps of conv1 (after the ReLU) for one image."""
    saved = {}
    def hook(layer, inputs, output):
        saved["maps"] = output[0].detach()
    h = model.features[1].register_forward_hook(hook)    # features[1] = ReLU after conv1
    with torch.no_grad():
        model(preprocess(img).unsqueeze(0).to(device))
    h.remove()
    return saved["maps"]

img = photos[0]      # the zebra; try photos[1] for the plane
n_maps = 16          # how many feature maps to show per network

fig, axes = plt.subplots(2, n_maps // 2 + 1, figsize=(16, 4.5))
for row, (net_name, model) in enumerate(networks.items()):
    maps = conv1_maps(model, img)
    axes[row, 0].imshow(what_network_sees(img))
    axes[row, 0].set_ylabel(net_name, fontsize=11)
    for f in range(n_maps // 2):
        axes[row, f + 1].imshow(maps[f].numpy(), cmap="gray")
        axes[row, f + 1].set_title(f"filter {f}", fontsize=7)
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("conv1 feature maps: trained (top) vs untrained (bottom)")
plt.tight_layout(); plt.show()

**Think about it:**
- Can you still see the zebra in the untrained network's maps? Why might that be, even with random filters?
- How do the trained maps differ? Do different filters pick up different things (edges, colours, stripes)?

## 5. Similarity matrices: photos and toys
So far we looked at single images. Now we ask: **which images does a network treat as similar?**

For each image, we take the activations of a layer: one long list of numbers. Two images are **similar** for the network if their activation patterns are similar. We measure the **dissimilarity** as `1 − correlation`:
- **0** = identical activation patterns,
- **1** = unrelated patterns,
- up to **2** = opposite patterns.

Doing this for every pair of images gives a **representational dissimilarity matrix (RDM)**. This is exactly the tool Cichy et al. used to compare networks with the brain.

Our images: two animals (zebra, panda) and two objects (plane, banana), each as a **photo** and as a **toy**.

In [ ]:
# The categories, in the order they appear in the matrices (animals first, then objects).
# The names must match the filenames: <category>_photo and <category>_toy
CATEGORIES = ["zebra", "panda", "plane", "banana"]
VERSIONS = ["photo", "toy"]

# Load the images in a fixed order: zebra_photo, zebra_toy, panda_photo, ...
names, images = [], []
for cat in CATEGORIES:
    for version in VERSIONS:
        path = next(IMAGE_DIR.glob(f"{cat}_{version}.*"))
        names.append(f"{cat} {version}")
        images.append(Image.open(path).convert("RGB"))

fig, axes = plt.subplots(1, len(images), figsize=(2.4 * len(images), 2.8))
for ax, img, name in zip(axes, images, names):
    ax.imshow(what_network_sees(img)); ax.set_title(name, fontsize=10); ax.axis("off")
plt.show()

### What does each network call these images?
First, the familiar view: the top-5 labels. Which photos and toys does the trained network recognise? Keep the answers in mind when you look at the similarity matrices.

In [ ]:
net_name = "trained"     # change to "untrained" to compare
show_top5(images, names, classify(networks[net_name], images), title=f"{net_name} AlexNet")

**Think about it:** which toys does the trained network recognise? Is the realistic plush panda easier than the felted zebra?

### Your turn first
**Before running the next cells**, predict: in the **trained** network's last layer, will the **zebra toy** be closer to the **zebra photo** or to the **other toys**? What about the untrained network?

In [ ]:
# The layers we compare: early, middle and late
LAYERS = {
    "conv1": ("features", 1),     # ReLU after the 1st convolution
    "conv5": ("features", 11),    # ReLU after the 5th (last) convolution
    "fc7":   ("classifier", 5),   # ReLU after the 2nd fully connected layer
}

def get_activations(model, imgs):
    """Return a dict: layer name -> array of shape (number of images, number of units)."""
    saved = {}
    def make_hook(name):
        def hook(layer, inputs, output):
            out = output.detach()
            saved[name] = out.reshape(out.shape[0], -1).numpy()   # one long vector per image
        return hook
    hooks = [getattr(model, part)[i].register_forward_hook(make_hook(name))
             for name, (part, i) in LAYERS.items()]
    batch = torch.stack([preprocess(img) for img in imgs]).to(device)
    with torch.no_grad():
        model(batch)
    for h in hooks:
        h.remove()
    return saved

def rdm(acts):
    """Dissimilarity (1 - correlation) between every pair of images."""
    return 1 - np.corrcoef(acts)

# RDMs for every network and every layer
rdms = {net_name: {layer: rdm(a) for layer, a in get_activations(model, images).items()}
        for net_name, model in networks.items()}

In [ ]:
def show_rdms(rdms, names, title=None):
    """Plot one RDM per network (rows) and layer (columns)."""
    nets, layers = list(rdms), list(next(iter(rdms.values())))
    fig, axes = plt.subplots(len(nets), len(layers), figsize=(4.6 * len(layers), 4.4 * len(nets)), squeeze=False)
    for r, net_name in enumerate(nets):
        for c, layer in enumerate(layers):
            ax = axes[r, c]
            m = rdms[net_name][layer]
            off_diag = m[~np.eye(len(m), dtype=bool)]           # all pairs of different images
            # Colour range from the pairs of different images, so small differences are visible
            im = ax.imshow(m, cmap="viridis", vmin=off_diag.min(), vmax=off_diag.max())
            ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=90, fontsize=8)
            ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8)
            ax.set_title(f"{net_name}: {layer}")
            fig.colorbar(im, ax=ax, fraction=0.046, label="dissimilarity")
    if title:
        fig.suptitle(title)
    plt.tight_layout(); plt.show()

show_rdms(rdms, names, title="Dark = similar, bright = dissimilar")

**How to read the matrices:** each cell is one pair of images. The diagonal is always 0 (an image compared with itself). Each matrix has its own colour scale (see the colour bar), so compare the **patterns**, not the colours, between matrices. The images are sorted by category, so if a network groups images **by category**, you see dark **2 × 2 blocks** along the diagonal. If it groups **animals vs objects**, you see a larger **4 × 4 block** in the top left.

**Think about it:**
- Do you see category blocks? In which network, and in which layer?
- In the trained network's `fc7`: is the zebra toy closer to the zebra photo, or to the other toys?
- Are the two animals closer to each other than to the objects? Or is the zebra closer to the panda simply because both are **black and white**?
- What does the untrained network group together? (Hint: look at the images. What do similar pairs have in common?)

## 6. Summary: category or version?
We summarise each matrix in two numbers:
- **same category, different version** (e.g. zebra photo vs zebra toy),
- **different category, same version** (e.g. zebra toy vs plane toy).

If the first is **lower** (more similar), the network groups by **category**. If the second is lower, it groups by **version** (photo vs toy).

In [ ]:
def summarise(m):
    same_cat, same_version = [], []
    for i, a in enumerate(names):
        for j, b in enumerate(names):
            if i >= j:
                continue                                  # each pair only once
            cat_a, ver_a = a.split(" ")
            cat_b, ver_b = b.split(" ")
            if cat_a == cat_b and ver_a != ver_b:
                same_cat.append(m[i, j])
            elif cat_a != cat_b and ver_a == ver_b:
                same_version.append(m[i, j])
    return np.mean(same_cat), np.mean(same_version)

layers = list(LAYERS)
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, net_name in zip(axes, networks):
    values = [summarise(rdms[net_name][layer]) for layer in layers]
    ax.plot(layers, [v[0] for v in values], marker="o", label="same category, different version")
    ax.plot(layers, [v[1] for v in values], marker="o", label="different category, same version")
    ax.set_title(f"{net_name} AlexNet"); ax.set_xlabel("layer")
axes[0].set_ylabel("mean dissimilarity (lower = more similar)")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Think about it:**
- In the trained network, does grouping by category get stronger in later layers? Does it ever beat grouping by version?
- What happens in the untrained network across layers?
- Cichy et al. found that the untrained network still matched the brain somewhat, but worse than the trained one, especially for **later** brain responses. Does your result fit with that?

**Try it:** change `LAYERS` to include other layers, e.g. `"conv3": ("features", 7)` or `"fc6": ("classifier", 2)`, and rerun section 5 and 6.

## 7. Discussion questions
1. The untrained network has the right architecture but has learned nothing. What can architecture alone explain, and what needs learning?
2. In Week 3, AlexNet labelled the toys as completely different things. Do the similarity matrices tell the same story, or is the zebra toy "closer to a zebra" inside the network than its label suggested?
3. Cichy et al. compared these matrices with **brain** data. What would you expect the matrix of a human's visual cortex to look like for our eight images? And of a 5-year-old's?